# 05 - Model validation and threshold selection

This notebook integrates the production pipeline on training data only. It creates an entity-level validation split, measures candidate recall, compares model families, optimizes thresholds for macro F0.5, and saves the selected artifacts. It does not run final test inference.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import sys
import time
import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if not SRC.exists(): SRC = PROJECT_ROOT / 'code' / 'business_entity_resolution' / 'src'
sys.path.insert(0, str(SRC.resolve()))
from config import default_config
from data_loader import load_training_data
from preprocessing import preprocess_sources
from blocking import generate_candidates, evaluate_blocking_recall
from features import fit_feature_transformers, transform_features
from model import train_model, predict_proba, model_summary, save_model
from validation import create_validation_split, build_training_pairs, evaluate_predictions, evaluate_per_entity, optimize_threshold, calculate_candidate_recall

CONFIG = default_config().resolved(PROJECT_ROOT)
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
MODEL_DIR = ARTIFACT_ROOT / 'models'; FEATURE_DIR = ARTIFACT_ROOT / 'features'; EXPERIMENT_DIR = ARTIFACT_ROOT / 'experiments'
for directory in [MODEL_DIR, FEATURE_DIR, EXPERIMENT_DIR]: directory.mkdir(parents=True, exist_ok=True)
MAX_PAIRS = 250_000
MODEL_TYPES = [CONFIG.model.model_type, 'extra_trees', 'hist_gradient_boosting']


In [ ]:
training = load_training_data(CONFIG)
sources = preprocess_sources(training.sources, CONFIG.schema, CONFIG.normalization)
source_names = list(sources)
reference_source = source_names[0]
reference = sources[reference_source]
candidate_sources = {name: sources[name] for name in source_names[1:]}
truth = training.ground_truth
split = create_validation_split(reference, truth, validation_fraction=CONFIG.validation.validation_fraction, random_state=CONFIG.validation.random_state, stratify_by_match_presence=CONFIG.validation.stratify_by_match_presence)
print('Train references:', len(split.train_reference), 'Validation references:', len(split.validation_reference))


In [ ]:
# Candidate generation is identical for both partitions; only the reference IDs are split.
block_result = generate_candidates(reference, candidate_sources, CONFIG, reference_source=reference_source)
train_pairs = block_result.pairs[block_result.pairs.reference_entity_id.astype(str).isin(set(split.train_ids))].head(MAX_PAIRS).reset_index(drop=True)
valid_pairs = block_result.pairs[block_result.pairs.reference_entity_id.astype(str).isin(set(split.validation_ids))].head(MAX_PAIRS).reset_index(drop=True)
train_labeled = build_training_pairs(train_pairs, truth)
valid_labeled = build_training_pairs(valid_pairs, truth)
candidate_recall = calculate_candidate_recall(valid_pairs, truth)
print('Train pairs:', len(train_pairs), 'Validation pairs:', len(valid_pairs))
print('Validation candidate recall:', candidate_recall)


In [ ]:
# Fit TF-IDF on training records and reuse the same transformers for validation.
transformers = fit_feature_transformers(split.train_reference, candidate_sources, CONFIG)
train_features = transform_features(train_pairs, split.train_reference, candidate_sources, transformers, CONFIG)
valid_features = transform_features(valid_pairs, split.validation_reference, candidate_sources, transformers, CONFIG)
feature_names = train_features.feature_names
print('Feature count:', len(feature_names))


In [ ]:
experiments = []
trained_models = {}
threshold_tables = {}
for model_type in dict.fromkeys(MODEL_TYPES):
    started = time.perf_counter()
    model_config = replace(CONFIG.model, model_type=model_type)
    try:
        fitted = train_model(train_features.matrix, train_labeled.target.to_numpy(), model_config, feature_names=feature_names, transformer_metadata={'feature_names': list(feature_names), 'fit_scope': 'training_records_only'}, random_state=CONFIG.runtime.random_seed)
        scores = predict_proba(fitted, valid_features.matrix)
        scored = valid_pairs.copy(); scored['score'] = scores
        threshold, threshold_table = optimize_threshold(scored, truth, score_column='score', minimum=CONFIG.threshold_search.minimum, maximum=CONFIG.threshold_search.maximum, step=CONFIG.threshold_search.step, reference_ids=split.validation_ids)
        scored['predicted_match'] = scored.score >= threshold
        metrics = evaluate_predictions(scored, truth, reference_ids=split.validation_ids)
        elapsed = time.perf_counter() - started
        experiments.append({'experiment_id': f'{model_type}_{len(experiments):03d}', 'feature_version': 'production_features', 'model': model_type, 'hyperparameters': json.dumps(fitted.hyperparameters, default=str, sort_keys=True), 'threshold': threshold, 'precision': metrics['precision'], 'recall': metrics['recall'], 'f0.5': metrics['f0.5'], 'candidate_recall': candidate_recall, 'false_merge_rate': metrics['false_merge_rate'], 'singleton_metrics': json.dumps({'singleton_accuracy': metrics['singleton_accuracy']}), 'runtime': elapsed, 'notes': 'validation entity split; threshold selected on validation F0.5'})
        trained_models[model_type] = (fitted, threshold, scored, threshold_table)
    except Exception as exc:
        experiments.append({'experiment_id': f'{model_type}_{len(experiments):03d}', 'feature_version': 'production_features', 'model': model_type, 'hyperparameters': '{}', 'threshold': None, 'precision': None, 'recall': None, 'f0.5': None, 'candidate_recall': candidate_recall, 'false_merge_rate': None, 'singleton_metrics': '{}', 'runtime': time.perf_counter() - started, 'notes': f'unavailable: {exc}'})

experiment_table = pd.DataFrame(experiments).sort_values(['f0.5', 'precision'], ascending=False, na_position='last', kind='mergesort')
display(experiment_table)

In [ ]:
available = experiment_table[experiment_table.f0_5.notna()] if 'f0_5' in experiment_table.columns else experiment_table[experiment_table['f0.5'].notna()]
if available.empty: raise RuntimeError('No model completed validation')
selected_type = str(available.iloc[0]['model'])
selected_model, selected_threshold, selected_scores, selected_threshold_table = trained_models[selected_type]
selected_diagnostics = evaluate_per_entity(selected_scores, truth, reference_ids=split.validation_ids)
display(selected_diagnostics.head())
print('Selected model:', selected_type)
print('Selected threshold:', selected_threshold)
print('Selected validation F0.5:', float(available.iloc[0]['f0.5']))

In [ ]:
# Persist selected artifacts and measured experiment evidence.
selected_model_path = MODEL_DIR / 'selected_model.joblib'
transformer_path = FEATURE_DIR / 'selected_feature_transformers.joblib'
save_model(selected_model, selected_model_path)
joblib.dump(transformers, transformer_path)
experiment_path = EXPERIMENT_DIR / 'model_experiments.csv'
experiment_table.to_csv(experiment_path, index=False)
threshold_path = EXPERIMENT_DIR / f'{selected_type}_threshold_search.csv'
selected_threshold_table.to_csv(threshold_path, index=False)
metadata = {'selected_model': selected_type, 'threshold': float(selected_threshold), 'feature_names': list(feature_names), 'validation_metrics': available.iloc[0].to_dict(), 'model_summary': model_summary(selected_model), 'candidate_recall': candidate_recall, 'multi_match_behavior': 'retain every candidate above threshold; do not force one match', 'singleton_behavior': 'empty predictions are scored as correct for true zero-match entities'}
(EXPERIMENT_DIR / 'selected_configuration.json').write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')
print('Saved model:', selected_model_path)
print('Saved transformers:', transformer_path)
print('Saved experiment log:', experiment_path)

## FINAL CONFIGURATION

The executed cells above populate this section from measured validation evidence. The selected configuration must be copied into the reproducible pipeline only after reviewing `artifacts/experiments/model_experiments.csv` and the threshold-search table.

- **Preprocessing:** production `preprocessing.py` configuration from `CONFIG.normalization`; original and normalized values are retained.
- **Blocking:** production `CONFIG.blocking` multi-stage union; candidate recall is recorded before model evaluation.
- **Features:** production feature order saved in `selected_configuration.json`; TF-IDF transformers are fitted on training data and reused.
- **Model:** selected by measured validation macro F0.5, not accuracy.
- **Hyperparameters:** saved in the selected model artifact and experiment CSV.
- **Threshold:** selected by configured threshold search and validation F0.5 with conservative constraints.
- **Multi-match behavior:** retain zero, one, or many supported matches; never force one match.
- **Singleton rule:** a true zero-match reference receives credit only when its final prediction is empty.

No metric or model superiority should be claimed until this notebook has been executed and the saved experiment tables reviewed.